In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.downloader as api

# ==========================================
# BƯỚC 1: TẢI TOÀN BỘ VĂN BẢN VÀ TẠO MA TRẬN
# ==========================================
print("1. Đang tải dữ liệu 20 Newsgroups ...")
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
corpus = newsgroups.data 

# 1.1 Bag of Words (BoW)
print("-> Đang tạo Bag of Words (BoW)...")
vectorizer_bow = CountVectorizer(max_features=5000, stop_words='english')
X_bow = vectorizer_bow.fit_transform(corpus)
print(f"Kích thước ma trận BoW: {X_bow.shape}") 

# In góc ma trận BoW
vocab_10_words = vectorizer_bow.get_feature_names_out()[:10]
dense_bow_slice = X_bow[:5, :10].toarray() 
df_bow_matrix = pd.DataFrame(dense_bow_slice, columns=vocab_10_words)
print("\n--- GÓC NHỎ CỦA MA TRẬN BOW (Đếm số lần xuất hiện) ---")
print(df_bow_matrix)

# 1.2 TF-IDF
print("\n-> Đang tạo TF-IDF...")
vectorizer_tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = vectorizer_tfidf.fit_transform(corpus)
print(f"Kích thước ma trận TF-IDF: {X_tfidf.shape}")

# In góc ma trận TF-IDF
dense_tfidf_slice = X_tfidf[:5, :10].toarray() 
df_tfidf_matrix = pd.DataFrame(dense_tfidf_slice, columns=vocab_10_words)
print("\n--- GÓC NHỎ CỦA MA TRẬN TF-IDF (Điểm số trọng số) ---")
print(df_tfidf_matrix)

# ==========================================
# BƯỚC 2: TẢI MODEL VÀ TÌM TỪ GẦN NGHĨA
# ==========================================
print("\n2. Kiểm tra và tải các mô hình Word Embeddings...")

# Dùng mẹo globals() để không bị load lại nếu bấm chạy nhiều lần
if 'w2v_model' not in globals():
    print("-> Đang load Word2Vec...")
    w2v_model = api.load("word2vec-google-news-300")
else:
    print("-> Word2Vec đã có sẵn trong RAM!")

if 'glove_model' not in globals():
    print("-> Đang load GloVe...")
    glove_model = api.load("glove-wiki-gigaword-100")
else:
    print("-> GloVe đã có sẵn trong RAM!")

if 'fasttext_model' not in globals():
    print("-> Đang load FastText...")
    fasttext_model = api.load("fasttext-wiki-news-subwords-300")
else:
    print("-> FastText đã có sẵn trong RAM!")

print("\n3. Bắt đầu so sánh khả năng tìm từ gần nghĩa (và lỗi OOV)...")

words_to_test = [
    'king', 'computer', 'bank', 'apple', 'happy', 
    'car', 'automobile', 'playing', 'unhappiness', 
    'coronavirused', 'dog'
]

def get_similar_words(model, word):
    try:
        similar = model.most_similar(word, topn=5)
        return ", ".join([w[0] for w in similar])
    except KeyError:
        return "[LỖI OOV - Không có trong từ vựng]"

results = []
for w in words_to_test:
    results.append({
        "Từ gốc": w,
        "Word2Vec": get_similar_words(w2v_model, w),
        "GloVe": get_similar_words(glove_model, w),
        "FastText": get_similar_words(fasttext_model, w)
    })

df_results = pd.DataFrame(results)
print("\n--- BẢNG SO SÁNH 5 TỪ GẦN NGHĨA NHẤT ---")
# Cấu hình hiển thị bảng Pandas đẹp và gọn gàng hơn
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
display(df_results)


1. Đang tải dữ liệu 20 Newsgroups ...
-> Đang tạo Bag of Words (BoW)...
Kích thước ma trận BoW: (11314, 5000)

--- GÓC NHỎ CỦA MA TRẬN BOW (Đếm số lần xuất hiện) ---
   00  000  01  02  03  04  040  05  06  07
0   0    0   0   0   0   0    0   0   0   0
1   0    0   0   0   0   0    0   0   0   0
2   0    0   0   0   0   0    0   0   0   0
3   0    0   0   0   0   0    0   0   0   0
4   0    0   0   0   0   0    0   0   0   0

-> Đang tạo TF-IDF...
Kích thước ma trận TF-IDF: (11314, 5000)

--- GÓC NHỎ CỦA MA TRẬN TF-IDF (Điểm số trọng số) ---
    00  000   01   02   03   04  040   05   06   07
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

2. Kiểm tra và tải các mô hình Word Embeddings...
-> Word2Vec đã có sẵn trong RAM!
-> GloVe đã có sẵn trong RAM!
-> FastText đã có sẵn tro

,Từ gốc,Word2Vec,GloVe,FastText
0,king,"kings, queen, monarch, crown_prince, prince","prince, queen, son, brother, monarch","king-, boy-king, queen, prince, kings"
1,computer,"computers, laptop, laptop_computer, Computer, com_puter","computers, software, technology, pc, hardware","computers, non-computer, mini-computer, micro-computer, super-computer"
2,bank,"banks, banking, Bank, lender, banker","banks, banking, credit, investment, financial","banks, bank-, banking, bank., bank-to-bank"
3,apple,"apples, pear, fruit, berry, pears","microsoft, ibm, intel, software, dell","apples, pear, peach, fruit, apple-"
4,happy,"glad, pleased, ecstatic, overjoyed, thrilled","'m, feel, 're, i, 'll","happpy, happy-, happy-happy, unhappy, happier"
5,car,"vehicle, cars, SUV, minivan, truck","vehicle, truck, cars, driver, driving","cars, vehicle, non-car, automobile, super-car"
6,automobile,"auto, automobiles, automotive, Automobile, tycoon_Edsel_Ford","auto, motor, automotive, automobiles, car","automobiles, car, automotive, automobile-related, automobility"
7,playing,"play, played, Playing, plays, game","played, play, plays, player, players","play, played, palying, playacting, plays"
8,unhappiness,"dissatisfaction, displeasure, discontent, discontentment, frustration","dissatisfaction, ambivalence, uneasiness, impatience, unease","dissatisfaction, misery, discontent, happiness, frustration"
9,coronavirused,[LỖI OOV - Không có trong từ vựng],[LỖI OOV - Không có trong từ vựng],[LỖI OOV - Không có trong từ vựng]
